# A readable report

The orders from `../data/orders.json` as an aligned table, built from ordinary format strings such as `{:>10}` and `{:>7.1}`. The steps of `demos/03_report.rn`.

In [1]:
fn dollars(cents) { format!("{}.{:02}", cents / 100, cents % 100) }

fn bar(n) {
    let s = "";
    for i in 0..n { s += "#"; }
    s
}

One row per region: [region, orders, shipped units, shipped cents]. Order counts include every status. The cell's value is the rows, sorted by region.

In [2]:
let orders = json::parse(fs::read("../data/orders.json")?)?;
let rows = [];
for o in orders {
    let row = None;
    for r in rows { if r[0] == o.region { row = Some(r); } }
    let row = match row { Some(r) => r, None => { let r = [o.region, 0, 0, 0]; rows.push(r); r } };
    row[1] = row[1] + 1;
    if o.status == "shipped" { row[2] = row[2] + o.qty; row[3] = row[3] + o.qty * o.unit_cents; }
}
rows.sort_by(|a, b| a[0].cmp(b[0]));
rows

[["east", 3, 11, 14497], ["north", 3, 13, 24500], ["south", 4, 5, 9995], ["west", 2, 0, 0]]

Totals, and the largest revenue, which sets the bar scale. With no shipped revenue, shares are undefined and shown as `-`.

In [3]:
let (orders_n, units, total, top) = (0, 0, 0, 0);
for r in rows { orders_n += r[1]; units += r[2]; total += r[3]; if r[3] > top { top = r[3]; } }
let share = |cents| if total == 0 { "-" } else { format!("{:.1}%", cents as f64 * 100.0 / total as f64) };
[orders_n, units, total]

[12, 29, 48992]

In [4]:
println!("{:<8}{:>7}{:>7}{:>10}{:>8}", "region", "orders", "units", "revenue", "share");
println!("{}", "--------------------------------------------------------------");
for r in rows {
    let width = if top == 0 { 0 } else { (r[3] * 20 + top / 2) / top };
    let line = format!("{:<8}{:>7}{:>7}{:>10}{:>8}", r[0], r[1], r[2], dollars(r[3]), share(r[3]));
    println!("{}", if width > 0 { line + "  " + bar(width) } else { line });
}
println!("{}", "--------------------------------------------------------------");
println!("{:<8}{:>7}{:>7}{:>10}{:>8}", "total", orders_n, units, dollars(total), share(total));

region   orders  units   revenue   share
--------------------------------------------------------------
east          3     11    144.97   29.6%  ############
north         3     13    245.00   50.0%  ####################


south         4      5     99.95   20.4%  ########
west          2      0      0.00    0.0%
--------------------------------------------------------------
total        12     29    489.92  100.0%
